# 🪙 เครื่องมือดึงข่าวทองคำ Thai PBS สู่ Quant JSON Schema 1.0
ดัดแปลงให้ใช้งานง่ายที่สุด: **ตั้งค่าแค่วันที่ใน Cell ด้านล่างแล้วกด Run All ได้ทันที**
- **1 ข่าว = 1 JSON Object** สมบูรณ์ตาม Schema 1.0 (แยก 4 กลุ่ม: `source/metadata`, `content`, `classification`, `extracted_facts`)
- บันทึกไฟล์อัตโนมัติ 4 รูปแบบ: `.json` (JSON Array), `.jsonl` (JSON Lines), `.csv`, และ `.md`

In [11]:
# ========================================================
# ⚙️ ตั้งค่าวันที่ที่ต้องการดึงข่าว (ผู้ใช้แก้ไขเฉพาะ Cell นี้)
# ========================================================
START_DATE = "2026-10-15"   # วันที่เริ่มต้น (YYYY-MM-DD)
END_DATE   = "2026-10-23"   # วันที่สิ้นสุด (YYYY-MM-DD)
MODE       = "gold"         # "gold" = ข่าวทองทั้งหมด | "trading" = เฉพาะรายงานราคาและวิเคราะห์
EXPORT     = True           # True = บันทึก JSON, JSONL, CSV และ Markdown อัตโนมัติ


In [12]:
# ponytail: all-in-one minimal engine producing quant-ready JSON schema (thaipbs-apis-docs)
import re
import time
import html
import json
import os
import requests
import pandas as pd
from datetime import datetime, timezone
from pythainlp.tokenize import word_tokenize
from pythainlp.util import Trie, normalize, thai_digit_to_arabic_digit
from pythainlp.corpus import thai_words
from IPython.display import display

BASE_URL = "https://api.thaipbs.net"
TZ = "Asia/Bangkok"
CACHE_FILE = "thaipbs_cache.json"
SCAN_MAX_PAGES = 360
TAG_MAX_PAGES = 30
MAX_DETAIL = 3000
REQUEST_DELAY = 0.05

COMMA_TAGS = "ราคาทองวันนี้,ราคาทอง,ข่าวราคาทอง,สรุปราคาทอง,ราคาทองคำตลาดโลก,ราคาทองคำ,ทองคำ"
SUPPORTED_TAG_SITES = ["now", "theactive", "policywatch", "verify"]

session = requests.Session()
session.headers.update({"User-Agent": "gold-news-quick/1.0", "Accept": "application/json"})

# Cache management
_DETAIL_CACHE = {}
if os.path.exists(CACHE_FILE):
    try:
        with open(CACHE_FILE, encoding="utf-8") as f:
            _DETAIL_CACHE = json.load(f)
    except Exception:
        _DETAIL_CACHE = {}

def save_cache():
    try:
        with open(CACHE_FILE, "w", encoding="utf-8") as f:
            json.dump(_DETAIL_CACHE, f, ensure_ascii=False)
    except Exception:
        pass

def api_get(path, params=None, retries=2):
    url = path if path.startswith("http") else f"{BASE_URL}{path}"
    for attempt in range(1, retries + 1):
        try:
            r = session.get(url, params=params, timeout=20)
            r.raise_for_status()
            return r.json()
        except Exception:
            if attempt == retries:
                return None
            time.sleep(attempt * 0.4)

def get_detail(site, p):
    path = p.get("restUrl") or f"/v1/{site}/posts/{p.get('id')}"
    if path in _DETAIL_CACHE:
        return _DETAIL_CACHE[path]
    d = api_get(path)
    time.sleep(REQUEST_DELAY)
    if not isinstance(d, dict):
        return None
    if isinstance(d.get("data"), dict) and "title" not in d:
        d = d["data"]
    keep = {k: d.get(k) for k in ("id", "title", "abstract", "content", "publishTime", "createTime")}
    _DETAIL_CACHE[path] = keep
    return keep

# Date Parsing
ISO_KEYS = ("publishTime", "publishedAt", "published_at", "createTime", "lastUpdateTime", "date")
THAI_MONTHS = {
    "ม.ค.": 1, "ก.พ.": 2, "มี.ค.": 3, "เม.ย.": 4, "พ.ค.": 5, "มิ.ย.": 6,
    "ก.ค.": 7, "ส.ค.": 8, "ก.ย.": 9, "ต.ค.": 10, "พ.ย.": 11, "ธ.ค.": 12,
    "มกราคม": 1, "กุมภาพันธ์": 2, "มีนาคม": 3, "เมษายน": 4, "พฤษภาคม": 5, "มิถุนายน": 6,
    "กรกฎาคม": 7, "สิงหาคม": 8, "กันยายน": 9, "ตุลาคม": 10, "พฤศจิกายน": 11, "ธันวาคม": 12,
}
_M_ALT = "|".join(sorted((re.escape(m) for m in THAI_MONTHS), key=len, reverse=True))
THAI_DATE_RX = re.compile(rf"(\d{{1,2}})\s*({_M_ALT})\s*(\d{{2,4}})")
URL_DATE_RX = re.compile(r"(?<!\d)(20\d{2})(0[1-9]|1[0-2])(0[1-9]|[12]\d|3[01])(?!\d)")

def _to_ts(v):
    try:
        ts = pd.to_datetime(v, errors="coerce")
        if pd.isna(ts):
            return None
        return ts.tz_localize(TZ) if ts.tzinfo is None else ts.tz_convert(TZ)
    except Exception:
        return None

def resolve_date(p, detail=None, body=""):
    for src in (p, detail):
        if src:
            for k in ISO_KEYS:
                if src.get(k):
                    ts = _to_ts(src[k])
                    if ts is not None:
                        return ts, f"api:{k}"
    canon = (p or {}).get("canonical", "")
    m = URL_DATE_RX.search(str(canon).rstrip("/").rsplit("/", 1)[-1])
    if m:
        try:
            return pd.Timestamp(int(m[1]), int(m[2]), int(m[3]), tz=TZ), "url"
        except ValueError:
            pass
    m_t = THAI_DATE_RX.search((body or "")[:800])
    if m_t:
        try:
            y = int(m_t[3])
            y = y + 2500 if y < 100 else y
            y = y - 543 if y > 2400 else y
            return pd.Timestamp(y, THAI_MONTHS[m_t[2]], int(m_t[1]), tz=TZ), "content"
        except Exception:
            pass
    return None, None

# NLP & Gold Tokenization
GOLD_TOKENS = {"ทองคำ", "ทองคำแท่ง", "ทองรูปพรรณ", "ทองแท่ง", "ราคาทอง", "ราคาทองคำ", "ทองสปอต", "ร้านทอง", "gold", "xauusd", "xau"}
GOLD_CONTEXT = {"ซื้อ", "ขาย", "ราคา", "ร้าน", "ออม", "ลงทุน", "ตลาด", "เก็งกำไร", "แท่ง", "รูปพรรณ", "สปอต", "ค้า", "พุ่ง", "ร่วง", "ปรับ", "ทะลุ", "นิวไฮ", "วันนี้", "ขึ้น", "ลง"}
FALSE_FRIENDS = {"ทองคำเปลว", "ผงทองคำ", "ทองหล่อ", "ทองหยิบ", "ทองหยอด", "ฝอยทอง", "ทองม้วน", "ทองพลุ", "ทองแดง", "ทองเหลือง", "ทองขาว", "ทองกวาว", "ทองผาภูมิ", "ทองอวบ", "ทองโหลง", "ทองทิพย์"}
TRIE = Trie(set(thai_words()) | GOLD_TOKENS | FALSE_FRIENDS | {"ทอง"})
_ZW = re.compile(r"[\u200b-\u200d\u2060\ufeff]")

def clean_article(raw):
    if not raw:
        return ""
    t = re.sub(r"(?is)<(script|style)[^>]*>.*?</\1>", " ", str(raw))
    t = re.sub(r"(?i)<br\s*/?>|</(p|div|li|h[1-6]|tr|ul|ol|blockquote|section|article)>", "\n", t)
    t = re.sub(r"(?s)<[a-zA-Z/!][^>]*>", " ", t)
    t = _ZW.sub("", html.unescape(t)).replace("\xa0", " ")
    lines = []
    for ln in t.split("\n"):
        ln = re.sub(r"[ \t]+", " ", ln).strip()
        if ln and not re.search(r"^\s*อ่านข่าว(?:ที่เกี่ยวข้อง|เพิ่มเติม)", ln):
            lines.append(thai_digit_to_arabic_digit(normalize(ln)))
    return "\n\n".join(lines)

def tokenize(text):
    toks = word_tokenize(text, custom_dict=TRIE, engine="newmm", keep_whitespace=False)
    return [t.lower() for t in toks if t.strip()]

def gold_hits(tokens):
    n = 0
    for i, tok in enumerate(tokens):
        if tok in GOLD_TOKENS:
            n += 1
        elif tok == "ทอง":
            neigh = tokens[max(0, i - 1):i] + tokens[i + 1:i + 2]
            if any(x in GOLD_CONTEXT for x in neigh):
                n += 1
    return n

# Price & Market Extraction
def to_num(s):
    v = float(str(s).replace(",", ""))
    return int(v) if v == int(v) else v

QUOTE_TOKEN = re.compile(r"(?P<round>ครั้งที่\s*\d+)|(?P<label>ทองคำแท่ง|ทองแท่ง|ทองรูปพรรณ|ทองคำรูปพรรณ|รูปพรรณ)|(?P<side>รับซื้อ|ขายออก)[^\d]{0,15}(?P<num>\d{1,3}(?:,\d{3})+(?:\.\d+)?)")
SPOT_RX = re.compile(r"(\d{1,3}(?:,\d{3})*(?:\.\d+)?)\s*ดอลลาร์(?:สหรัฐ)?ฯ?\s*(?:ต่อ|/)\s*(?:ทรอย)?ออนซ์")
TARGET_WORDS = ("แตะ", "ลุ้น", "เป้า", "ทะลุ", "ไปที่", "ขึ้นไป", "เหนือ", "ต่ำกว่า", "หลุด", "แนวรับ", "แนวต้าน", "ตัดขาดทุน")
LEVEL_RX = re.compile(r"(?:แนวรับ|แนวต้าน|ตัดขาดทุน|ขายทำกำไร|เป้าหมาย|เป้า|กรอบ(?:ราคา)?|stop\s*loss|take\s*profit)[^\d]{0,60}\d[\d,\.]*(?:\s*-\s*\d[\d,\.]*)?\s*(?:ดอลลาร์|บาท)?(?:ต่อ|/)?(?:ออนซ์)?", re.I)
UP_WORDS = ("พุ่ง", "ขึ้น", "บวก", "ทะลุ", "นิวไฮ", "ฟื้น", "หนุน", "แพงขึ้น")
DOWN_WORDS = ("ดิ่ง", "ร่วง", "ลบ", "ลง", "หลุด", "กด", "ผ่อน", "ถูกลง", "อ่อน")
DRIVERS = {
    "เฟด/ดอกเบี้ย": ("เฟด", "ดอกเบี้ย", "fomc", "fed "),
    "เงินเฟ้อ": ("เงินเฟ้อ", "cpi", "pce"),
    "ดอลลาร์สหรัฐ": ("ดัชนีดอลลาร์", "ค่าเงินดอลลาร์", "dxy", "ดอลลาร์แข็ง", "ดอลลาร์อ่อน"),
    "บอนด์ยีลด์": ("บอนด์ยีลด์", "พันธบัตร", "bond yield"),
    "น้ำมัน/พลังงาน": ("น้ำมัน", "พลังงาน"),
    "ธนาคารกลางซื้อทอง": ("ธนาคารกลาง",),
    "ภูมิรัฐศาสตร์": ("ตะวันออกกลาง", "สงคราม", "ขัดแย้ง", "ภูมิรัฐศาสตร์", "อิหร่าน", "ยูเครน"),
    "กองทุน/ETF": ("spdr", "etf", "กองทุนทอง"),
}

def analyze_article(title, article):
    body = re.sub(r"\s+", " ", article or "").strip()
    t_tok, b_tok = tokenize(title), tokenize(body)
    t_hits, b_hits = gold_hits(t_tok), gold_hits(b_tok)
    
    signals = [k for k, rx in {
        "baht": re.compile(r"บาทละ\s*\d", re.I),
        "ounce": re.compile(r"ออนซ์|gold\s*spot|ทองสปอต", re.I),
        "quote": re.compile(r"(?:รับซื้อ|ขายออก)[^\d]{0,15}\d{2,3},\d{3}", re.I),
        "assoc": re.compile(r"สมาคมค้าทอง", re.I),
    }.items() if rx.search(f"{title} {body}")]
    
    drivers = [k for k, kws in DRIVERS.items() if any(w in body.lower() for w in kws)]
    levels = [re.sub(r"\s+", " ", m.group(0)).strip() for m in list(LEVEL_RX.finditer(body))[:5]]
    
    if t_hits < 1 or b_hits < 2:
        news_type = "off_topic"
    elif len(signals) >= 2 or re.search(r"สรุปราคา|เช้านี้|ปิดตลาด|เปิดตลาด", title):
        news_type = "price_report"
    elif drivers or re.search(r"จับตา|ปัจจัย|วิเคราะห์|แนวโน้ม|คาด|ลุ้น|เฟด", title):
        news_type = "market_analysis"
    else:
        news_type = "other_gold"
        
    report_type = "close" if re.search(r"สรุปราคา|ปิดตลาด", title) else "morning" if re.search(r"เช้านี้|เปิดตลาด", title) else None
    
    # Parse quote sets
    sets, cur, label = [], {}, "bar"
    for m in QUOTE_TOKEN.finditer(body):
        if m.group("round"):
            if cur: sets.append(cur)
            cur, label = {}, "bar"
        elif m.group("label"):
            label = "orn" if "รูปพรรณ" in m.group("label") else "bar"
        else:
            key = f"{label}_{'buy' if m.group('side') == 'รับซื้อ' else 'sell'}"
            if key in cur:
                sets.append(cur)
                cur = {}
            cur[key] = to_num(m.group("num"))
    if cur: sets.append(cur)
    head = sets[0] if sets else {}
    
    # Spot USD
    spot = None
    for m in SPOT_RX.finditer(body):
        before = body[max(0, m.start() - 14): m.start()].lower()
        if not any(w in before for w in TARGET_WORDS):
            val = to_num(m.group(1))
            if val >= 500:
                spot = val
                break
                
    # Direction & change
    direction, change, n_changes = None, None, None
    m_c = re.search(r"([+\-–−])\s*(\d[\d,]*)\s*บาท", title)
    if m_c:
        change = int(m_c.group(2).replace(",", "")) * (1 if m_c.group(1) == "+" else -1)
    elif "ไม่เปลี่ยนแปลง" in title:
        change = 0
    if change is not None:
        direction = "up" if change > 0 else "down" if change < 0 else "flat"
    else:
        direction = "up" if any(w in title for w in UP_WORDS) else "down" if any(w in title for w in DOWN_WORDS) else None
        
    m_n = re.search(r"ผันผวน\s*(\d+)\s*ครั้ง", title)
    if m_n: n_changes = int(m_n.group(1))
    
    return {
        "news_type": news_type,
        "report_type": report_type,
        "direction": direction,
        "change_baht": change,
        "n_changes": n_changes,
        "bar_buy": head.get("bar_buy"),
        "bar_sell": head.get("bar_sell"),
        "orn_buy": head.get("orn_buy"),
        "orn_sell": head.get("orn_sell"),
        "spot_usd": spot,
        "drivers": drivers,
        "levels": levels,
        "head": head,
    }

# Schema Builder for Quant Research
def build_news_json(site, p, raw_content, article, ts, info):
    pid = str(p.get("id") or "")
    tags = [t.get("title", "") if isinstance(t, dict) else str(t) for t in (p.get("tags") or [])]
    cats = [c.get("title", "") if isinstance(c, dict) else str(c) for c in (p.get("categories") or [])]
    pub_iso = ts.isoformat() if ts is not None else None
    pub_utc = ts.tz_convert("UTC").strftime("%Y-%m-%dT%H:%M:%SZ") if ts is not None else None
    head = info.get("head", {})
    
    return {
        "schema_version": "1.0",
        "article_id": f"thaipbs:{site}:{pid}",
        "source": {
            "name": "Thai PBS",
            "site": site,
            "canonical_url": p.get("canonical"),
        },
        "metadata": {
            "title": re.sub(r"\s+", " ", p.get("title") or "").strip(),
            "language": "th",
            "published_at": pub_iso,
            "published_at_utc": pub_utc,
            "tags": [t for t in tags if t],
            "categories": [c for c in cats if c],
        },
        "content": {
            "text": article,
            "body_source": "content" if raw_content else "abstract",
            "content_status": "full_text" if raw_content else "abstract_only",
        },
        "classification": {
            "topic": "gold_market",
            "news_type": info["news_type"],
            "report_type": info["report_type"],
            "direction": info["direction"],
            "change_baht": info["change_baht"],
            "method": "rule_based",
        },
        "extracted_facts": {
            "thai_gold_bar": {
                "buy_price_baht": head.get("bar_buy"),
                "sell_price_baht": head.get("bar_sell"),
            },
            "thai_gold_ornament": {
                "buy_price_baht": head.get("orn_buy"),
                "sell_price_baht": head.get("orn_sell"),
            },
            "global_gold_spot": {
                "price_usd_per_ounce": info["spot_usd"],
            },
            "market_drivers": info["drivers"],
            "price_levels": info["levels"],
        },
        "provenance": {
            "retrieved_at": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
            "extraction_version": "1.0",
        },
    }

# Search & Collector
def search_and_export(start_date, end_date, mode="gold", export=True):
    start = pd.Timestamp(start_date, tz=TZ)
    end = pd.Timestamp(end_date, tz=TZ) + pd.Timedelta(days=1) - pd.Timedelta(seconds=1)
    print(f"🔍 ค้นหาข่าวทองคำ: {start.date()} ถึง {end.date()} | โหมด: {mode}")
    
    cands, seen = {}, set()
    
    # 1. Scan site 'news' (แหล่งข่าวราคาทองคำหลัก)
    print("  -> สแกนข่าวประจำวัน (news)...")
    for page in range(1, SCAN_MAX_PAGES + 1):
        data = api_get("/v1/news/posts", [("page", page), ("limit", 20)])
        posts = (data or {}).get("data") or []
        if not posts: break
        dates = []
        for p in posts:
            pid = p.get("id") or p.get("canonical")
            if pid in seen: continue
            seen.add(pid)
            ts, _ = resolve_date(p)
            if ts is not None: dates.append(ts)
            if ts is not None and ts > end: continue
            if gold_hits(tokenize(p.get("title") or "")) > 0:
                cands[("news", pid)] = p
        if dates and max(dates) < start:
            break
        tp = (data or {}).get("totalPages")
        if tp and page >= int(tp): break
        
    # 2. Tag query for supporting sites
    for site in SUPPORTED_TAG_SITES:
        for page in range(1, TAG_MAX_PAGES + 1):
            data = api_get(f"/v1/{site}/posts", [("page", page), ("limit", 20), ("tag", COMMA_TAGS)])
            posts = (data or {}).get("data") or []
            if not posts: break
            dates = []
            for p in posts:
                pid = p.get("id") or p.get("canonical")
                if pid in seen: continue
                seen.add(pid)
                ts, _ = resolve_date(p)
                if ts is not None: dates.append(ts)
                if ts is not None and ts > end: continue
                if gold_hits(tokenize(p.get("title") or "")) > 0:
                    cands[(site, pid)] = p
            if dates and max(dates) < start: break
            tp = (data or {}).get("totalPages")
            if tp and page >= int(tp): break

    print(f"  -> พบข่าวผู้สมัครเบื้องต้น: {len(cands)} รายการ")
    if not cands:
        print("❌ ไม่พบข่าวที่เข้าเงื่อนไขในช่วงเวลานี้")
        return pd.DataFrame(), []

    # 3. Fetch full article details via restUrl & build JSONs
    rows, json_items = [], []
    items = list(cands.items())[:MAX_DETAIL]
    for i, ((site, pid), p) in enumerate(items, 1):
        if i % 10 == 0 or i == len(items):
            print(f"  -> ดึงเนื้อหาแล้ว {i}/{len(items)} ชิ้น...")
        detail = get_detail(site, p)
        raw_content = ""
        if detail:
            raw_content = detail.get("content") or detail.get("abstract") or ""
        article = clean_article(raw_content or p.get("abstract") or "")
        ts, src = resolve_date(p, detail, article)
        if ts is None or ts < start or ts > end:
            continue
            
        title = re.sub(r"\s+", " ", p.get("title") or "").strip()
        info = analyze_article(title, article)
        
        # Check mode filtering
        target_types = {"price_report", "market_analysis"} if mode == "trading" else {"price_report", "market_analysis", "other_gold"}
        if info["news_type"] not in target_types:
            continue
            
        news_obj = build_news_json(site, p, raw_content, article, ts, info)
        json_items.append(news_obj)
        
        rows.append({
            "published": ts,
            "site": site,
            "title": title,
            "news_type": info["news_type"],
            "report_type": info["report_type"],
            "direction": info["direction"],
            "change_baht": info["change_baht"],
            "bar_buy": info["bar_buy"],
            "bar_sell": info["bar_sell"],
            "orn_buy": info["orn_buy"],
            "orn_sell": info["orn_sell"],
            "spot_usd": info["spot_usd"],
            "drivers": ", ".join(info["drivers"]),
            "canonical": p.get("canonical"),
            "article": article,
        })
    save_cache()
    
    if not rows:
        print("❌ ไม่พบข่าวที่อยู่ในช่วงวันที่กำหนดหลังจากตรวจสอบวันที่ในเนื้อหาแล้ว")
        return pd.DataFrame(), []
        
    df_all = pd.DataFrame(rows).sort_values("published", ascending=False).reset_index(drop=True)
    df = df_all.drop_duplicates(subset=["title"]).reset_index(drop=True)
    
    # Remove duplicates from json_items matching unique titles
    seen_titles = set()
    unique_jsons = []
    for j in sorted(json_items, key=lambda x: x["metadata"]["published_at"] or "", reverse=True):
        t = j["metadata"]["title"]
        if t not in seen_titles:
            seen_titles.add(t)
            unique_jsons.append(j)
    json_items = unique_jsons
    
    print(f"✅ สำเร็จ! ได้ข่าวทองคำตรงเป้าหมาย: {len(df)} รายการ\n")
    
    # 4. Export (JSON, JSONL, CSV, Markdown)
    if export and json_items:
        d0 = start.strftime("%Y%m%d")
        d1 = end.strftime("%Y%m%d")
        base_name = f"thaipbs_gold_{d0}_{d1}"
        
        # 4.1 JSON Array
        json_file = f"{base_name}.json"
        with open(json_file, "w", encoding="utf-8") as f:
            json.dump(json_items, f, ensure_ascii=False, indent=2)
            
        # 4.2 JSON Lines (JSONL) - ideal for streaming / quant processing
        jsonl_file = f"{base_name}.jsonl"
        with open(jsonl_file, "w", encoding="utf-8") as f:
            for item in json_items:
                f.write(json.dumps(item, ensure_ascii=False) + "\n")
                
        # 4.3 CSV
        df_exp = df.copy()
        df_exp["published"] = df_exp["published"].dt.strftime("%Y-%m-%d %H:%M")
        csv_file = f"{base_name}.csv"
        df_exp.to_csv(csv_file, index=False, encoding="utf-8-sig")
        
        # 4.4 Markdown
        md_file = f"{base_name}.md"
        md_lines = [f"# สรุปข่าวทองคำ Thai PBS ({d0} - {d1})", f"รวม {len(df)} ข่าว\n"]
        for _, r in df_exp.iterrows():
            md_lines.append(f"## {r['title']}")
            rep = f"/{r['report_type']}" if pd.notna(r.get('report_type')) and r['report_type'] else ""
            md_lines.append(f"- **เวลา**: {r['published']} | **แหล่ง**: {r['site']} | **ประเภท**: {r['news_type']}{rep}")
            md_lines.append(f"- **ต้นฉบับ**: {r['canonical']}")
            if pd.notna(r.get('bar_sell')):
                md_lines.append(f"- **ราคาทองแท่ง**: รับซื้อ {r['bar_buy']:,} / ขายออก {r['bar_sell']:,} บาท")
            if pd.notna(r.get('spot_usd')):
                md_lines.append(f"- **Gold Spot**: {r['spot_usd']:,} USD/oz")
            if r.get('drivers'):
                md_lines.append(f"- **ปัจจัย**: {r['drivers']}")
            md_lines.append(f"\n{r['article']}\n\n---\n")
        with open(md_file, "w", encoding="utf-8") as f:
            f.write("\n".join(md_lines))
            
        print(f"💾 บันทึกไฟล์เรียบร้อยแล้ว:")
        print(f"   - JSON  : {json_file}")
        print(f"   - JSONL : {jsonl_file}")
        print(f"   - CSV   : {csv_file}")
        print(f"   - MD    : {md_file}")
        
    # 5. Display table view & Example JSON
    cols = [c for c in ["published", "site", "news_type", "title", "change_baht", "bar_sell", "spot_usd", "drivers"] if c in df.columns]
    display(df[cols].head(15))
    
    if json_items:
        print("\n✨ ตัวอย่าง JSON Object ข่าวชิ้นแรก (Quant Schema 1.0):")
        print(json.dumps(json_items[0], indent=2, ensure_ascii=False))
        
    return df, json_items

# 🚀 รันดึงข่าวทันทีตามวันที่ที่ตั้งไว้ใน Cell 1
df, json_items = search_and_export(START_DATE, END_DATE, mode=MODE, export=EXPORT)


🔍 ค้นหาข่าวทองคำ: 2026-10-15 ถึง 2026-10-23 | โหมด: gold
  -> สแกนข่าวประจำวัน (news)...
  -> พบข่าวผู้สมัครเบื้องต้น: 30 รายการ
  -> ดึงเนื้อหาแล้ว 10/30 ชิ้น...
  -> ดึงเนื้อหาแล้ว 20/30 ชิ้น...
  -> ดึงเนื้อหาแล้ว 30/30 ชิ้น...
✅ สำเร็จ! ได้ข่าวทองคำตรงเป้าหมาย: 13 รายการ

💾 บันทึกไฟล์เรียบร้อยแล้ว:
   - JSON  : thaipbs_gold_20261015_20261023.json
   - JSONL : thaipbs_gold_20261015_20261023.jsonl
   - CSV   : thaipbs_gold_20261015_20261023.csv
   - MD    : thaipbs_gold_20261015_20261023.md


,published,site,news_type,title,change_baht,bar_sell,spot_usd,drivers
0,2026-10-23 11:55:42+07:00,news,price_report,"ราคา ""ทองคำ"" ดิ่ง หลุด 65,000 บาท ตลาดกังวลเงิ...",NaN,64100.0,4009.0,"เงินเฟ้อ, บอนด์ยีลด์, น้ำมัน/พลังงาน, ธนาคารกล..."
1,2026-10-22 18:26:57+07:00,news,price_report,สรุปราคาทองคำ 22 ต.ค.2569 ปิดตลาด ผันผวน 25 คร...,NaN,65350.0,NaN,"เฟด/ดอกเบี้ย, เงินเฟ้อ, น้ำมัน/พลังงาน, ภูมิรั..."
2,2026-10-22 17:42:29+07:00,news,price_report,น้ำมัน–บอนด์ยีลด์กดทองคำ จับตาธนาคารกลางแห่ซื้...,NaN,NaN,4060.0,"บอนด์ยีลด์, น้ำมัน/พลังงาน, ธนาคารกลางซื้อทอง,..."
3,2026-10-21 17:28:35+07:00,news,price_report,สรุปราคาทองคำ 21ต.ค.2569 ผันผวน 20 ครั้ง ไม่เป...,0.0,65600.0,NaN,"เฟด/ดอกเบี้ย, ดอลลาร์สหรัฐ, บอนด์ยีลด์, น้ำมัน..."
4,2026-10-21 10:06:02+07:00,news,price_report,"ราคา ""ทองคำ"" เช้านี้ + 50 บาท ""รูปพรรณ"" ขายออก...",50.0,65400.0,4118.0,"เฟด/ดอกเบี้ย, เงินเฟ้อ, บอนด์ยีลด์, น้ำมัน/พลั..."
5,2026-10-20 17:52:29+07:00,news,price_report,สรุปราคาทองคำ 20 ต.ค.2569 ปิดตลาด ผันผวน 21 คร...,NaN,65600.0,NaN,"เฟด/ดอกเบี้ย, เงินเฟ้อ, ดอลลาร์สหรัฐ, น้ำมัน/พ..."
6,2026-10-20 10:24:27+07:00,news,price_report,"ราคา ""ทองคำ"" เช้านี้ -400 บาท ""รูปพรรณ"" ขายออก...",-400.0,65850.0,4140.0,"เฟด/ดอกเบี้ย, เงินเฟ้อ, บอนด์ยีลด์, น้ำมัน/พลั..."
7,2026-10-19 17:43:09+07:00,news,price_report,สรุปราคาทองคำ 19 ต.ค.2569 ปิดตลาด ผันผวน 23 คร...,NaN,66250.0,NaN,"เฟด/ดอกเบี้ย, เงินเฟ้อ, บอนด์ยีลด์, น้ำมัน/พลั..."
8,2026-10-19 13:23:15+07:00,news,price_report,"จับตาประชุมดอกเบี้ย 28-29 ต.ค. ""ทองคำ"" ยังเผชิ...",NaN,NaN,4200.0,"เฟด/ดอกเบี้ย, เงินเฟ้อ, บอนด์ยีลด์, น้ำมัน/พลั..."
9,2026-10-16 18:10:34+07:00,news,price_report,สรุปราคาทองคำ 16 ต.ค.2569 ปิดตลาด ผันผวน 23 คร...,NaN,66100.0,NaN,"เฟด/ดอกเบี้ย, น้ำมัน/พลังงาน, ธนาคารกลางซื้อทอ..."



✨ ตัวอย่าง JSON Object ข่าวชิ้นแรก (Quant Schema 1.0):
{
  "schema_version": "1.0",
  "article_id": "thaipbs:news:571532",
  "source": {
    "name": "Thai PBS",
    "site": "news",
    "canonical_url": "https://www.thaipbs.or.th/news/content/571532"
  },
  "metadata": {
    "title": "ราคา \"ทองคำ\" ดิ่ง หลุด 65,000 บาท ตลาดกังวลเงินเฟ้อภาคพลังงาน ทองโลกสู่ขาลงต่อ",
    "language": "th",
    "published_at": "2026-10-23T11:55:42+07:00",
    "published_at_utc": "2026-10-23T04:55:42Z",
    "tags": [
      "ราคาทองวันนี้",
      "ราคาทอง",
      "ราคาทองคำแท่ง",
      "ราคาทองคำรูปพรรณ",
      "ทองคำแท่ง",
      "ทองคำรูปพรรณ",
      "แหวนทอง 1 สลึง ราคาวันนี้",
      "ราคาทองวันนี้ 1 สลึง ขาย",
      "ราคาทองวันนี้ เยาวราช",
      "สมาคมค้าทอง",
      "ราคาดอลลาร์วันนี้",
      "ข่าวราคาทอง",
      "สรุปราคาทอง",
      "ข่าววันนี้",
      "ข่าวเศรษฐกิจ"
    ],
    "categories": [
      "เศรษฐกิจ"
    ]
  },
  "content": {
    "text": "วันนี้ (23 ต.ค.2569) เว็บไซต์ฮั่วเซ่งเฮง วิเคราะสถานกา